# Phase VII — private-repository launcher

Use this launcher if `ardominguezm/painting-geometry` is private.

### One-time setup in Colab
1. Create a **fine-grained GitHub personal access token** with access only to `painting-geometry` and **Contents: Read-only**.
2. In Colab, open the **Secrets** panel (key icon) and add:
   - Name: `GITHUB_TOKEN`
   - Value: your token
   - Enable **Notebook access**.

Then run **Runtime → Run all**.

The token is read from Colab Secrets and is never written into the notebook, Drive, Git configuration, or command line.
This launcher clones the private branch securely and then executes the current Phase-VII workflow from the cloned repository.


In [ ]:
# 0. Secure private-repository setup
import os, sys, subprocess, shutil, json, tempfile, stat
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = "https://github.com/ardominguezm/painting-geometry.git"
BRANCH = "multiscale-corpus-analysis"
REPO_DIR = Path("/content/painting-geometry")

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Colab secret GITHUB_TOKEN is missing. "
        "Open the key/Secrets panel, add GITHUB_TOKEN, and enable Notebook access."
    ) from exc

if not GITHUB_TOKEN:
    raise RuntimeError("GITHUB_TOKEN is empty.")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# Use GIT_ASKPASS so the token does NOT appear in the clone URL or notebook output.
askpass = Path("/content/git_askpass_phase7.sh")
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    'esac\n'
)
askpass.chmod(0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"
env["GITHUB_TOKEN"] = GITHUB_TOKEN

proc = subprocess.run(
    ["git","clone","--depth","1","--branch",BRANCH,"--single-branch",REPO_URL,str(REPO_DIR)],
    text=True, capture_output=True, env=env
)

# Remove the temporary credential helper immediately.
try:
    askpass.unlink()
except FileNotFoundError:
    pass
env.pop("GITHUB_TOKEN", None)
GITHUB_TOKEN = None

if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(
        "Private GitHub clone failed. Check that the token has access to "
        "ardominguezm/painting-geometry and Contents: Read permission."
    )

subprocess.run(
    [sys.executable,"-m","pip","install","-q","-r",str(REPO_DIR/"requirements.txt")],
    check=True
)
subprocess.run(
    [sys.executable,"-m","pip","install","-q","ordpy>=1.2.0","kagglehub"],
    check=True
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Private repository cloned ✓")
print("Branch:", BRANCH)
print("Commit:", subprocess.check_output(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],text=True).strip())


In [ ]:
# 1. Execute the full Phase-VII workflow from the private clone

from IPython.display import display

workflow_path = REPO_DIR / "notebooks" / "11_phase7_full_artbench_confirmatory_colab.ipynb"
if not workflow_path.exists():
    raise FileNotFoundError(workflow_path)

workflow = json.loads(workflow_path.read_text(encoding="utf-8"))

# Recreate exactly the variables that the original setup cell would define,
# while keeping the authenticated clone we already made.
import tarfile, zipfile
from google.colab import files
import numpy as np, pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive/painting_geometry_phase7_full")
CACHE_DIR = DRIVE_ROOT / "cache"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "B90_G44_chunks"
RESULTS_DIR = DRIVE_ROOT / "results"
DATA_DIR = Path("/content/artbench_data")
EXTRACT_DIR = DATA_DIR / "imagefolder"

for p in [DRIVE_ROOT, CACHE_DIR, CHECKPOINT_DIR, RESULTS_DIR, DATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CACHE_ARTBENCH_TAR_IN_DRIVE = True
FEATURE_CHUNK_SIZE = 500
ORDINAL_CHECKPOINT_EVERY = 5000
N_PERMUTATIONS = 4999
N_BOOTSTRAP = 5000
COMMIT = subprocess.check_output(
    ["git","-C",str(REPO_DIR),"rev-parse","HEAD"], text=True
).strip()

print("Persistent root:", DRIVE_ROOT)
print("Existing B90/G44 chunks:", len(list(CHECKPOINT_DIR.glob("chunk_*.csv"))))

code_cells = [c for c in workflow["cells"] if c.get("cell_type") == "code"]

# Skip only the original unauthenticated setup cell.
to_run = []
for c in code_cells:
    src = "".join(c.get("source", []))
    if src.lstrip().startswith("# 0. Setup: Drive + repository + dependencies"):
        continue
    to_run.append(src)

print(f"Executing {len(to_run)} Phase-VII code cells...")
for i, src in enumerate(to_run, start=1):
    print("\n" + "="*72)
    print(f"PHASE VII CELL {i}/{len(to_run)}")
    print("="*72)
    exec(compile(src, f"<phase7-cell-{i}>", "exec"), globals(), globals())

print("\nPHASE VII COMPLETE ✓")


### Persistence

The heavy feature extraction checkpoints remain in:

`MyDrive/painting_geometry_phase7_full/checkpoints/`

If Colab disconnects, reopen this launcher and run it again. Completed chunks are detected and skipped.

At the end, the compact archive is:

`MyDrive/painting_geometry_phase7_full/painting_geometry_phase7_full_results_LIGHT.zip`
